# Análisis y modelo — Airbnb Price Prediction

Notebook técnico para limpiar el dataset de Kaggle, entrenar el modelo con `log_price` como objetivo y revisar las métricas principales del ajuste.

## Contexto
- Los datos provienen del dataset de Kaggle Airbnb Price Prediction.
- `raw.csv` es el archivo descargado del dataset original.
- `cleaned.csv` es la versión preparada para modelar.
- El objetivo del modelo es `log_price`, mientras que `price` se usa para interpretar los resultados en una escala más amigable.

## Requisitos
- Este notebook asume que las dependencias están instaladas (`requirements.txt`).
- Los datos salen de archivos locales en `Datos/`.
- Si no existe `cleaned.csv`, el notebook intenta crearlo a partir de `raw.csv`.
- El flujo está pensado para trabajar con el dataset de Kaggle Airbnb Price Prediction.

In [ ]:
# Imports
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Image, display

# Reutiliza funciones del proyecto
from Limpieza import load_data, clean_data
from Modelo import train_model
from Visualizaciones.plot_predictions import plot_predictions

plt.style.use('seaborn')

In [ ]:
# Rutas
data_dir = os.path.join('Datos')
raw_path = os.path.join(data_dir, 'raw.csv')
clean_path = os.path.join(data_dir, 'cleaned.csv')

# Si no existe el CSV limpio, intentar crearlo desde el raw (si existe)
if not os.path.exists(clean_path):
    if os.path.exists(raw_path):
        print('Limpiando el conjunto de datos crudo...')
        df_raw = load_data(raw_path)
        df_clean = clean_data(df_raw)
        os.makedirs(data_dir, exist_ok=True)
        df_clean.to_csv(clean_path, index=False)
        print(f'Datos limpios guardados en {clean_path}')
    else:
        raise FileNotFoundError(f'No se encontró el archivo en {raw_path}. Agrega tu dataset en esa ruta.')
else:
    print(f'Usando dataset limpio existente: {clean_path}')

In [ ]:
# Cargar datos limpios e inspeccionar
df = pd.read_csv(clean_path, low_memory=False)
if 'price' not in df.columns and 'log_price' in df.columns:
    df['price'] = np.exp(df['log_price'])
print('Forma:', df.shape)
display(df.head())
display(df[['log_price', 'price']].describe())

In [ ]:
# Entrenar modelo con el flujo actualizado
res = train_model(clean_path, target='log_price', model_out='model.joblib')

print('R² (log_price):', res['r2'])
print('R² ajustado (log_price):', res['adj_r2'])
print('MSE (log_price):', res['mse'])
print('RMSE (log_price):', res['rmse'])
print('RMSE (precio real):', res['price_rmse'])
print('MAE (precio real):', res['price_mae'])

print('\nVIF:')
display(res.get('vif'))

print('\nCoeficientes principales:')
display(res.get('coef_df').head(15))

In [ ]:
# Generar gráficas de predicción (guarda archivos y los muestra)
pred_csv = 'predictions.csv'
out_dir = 'Visualizaciones'
os.makedirs(out_dir, exist_ok=True)

pd.DataFrame({
    'price_true': res['price_true'],
    'price_pred': res['price_pred'],
    'log_price_true': res['y_test_log'],
    'log_price_pred': res['y_pred_log'],
}).to_csv(pred_csv, index=False)

plot_predictions(pred_csv, out_dir=out_dir)

display(Image(os.path.join(out_dir, 'predicted_vs_actual.png')))
display(Image(os.path.join(out_dir, 'residuals.png')))

## Conclusiones y próximos pasos
- El proyecto ya trabaja sobre el dataset real de Kaggle y no sobre el ejemplo pequeño anterior.
- El modelo usa `log_price` como objetivo y luego convierte los resultados a precio real para interpretarlos mejor.
- Las variables de capacidad y ubicación son las que más aportan al ajuste.
- La versión actual ya sirve como base sólida, aunque todavía puede mejorar con más ingeniería de variables o con modelos alternativos.